<a href="https://colab.research.google.com/github/nawanglhantso/cis3120-spring2026/blob/mp%2F03-industry-comparison-team-15/MP03_Notebook_team_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `15` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `Mahi Uddin`
- Travel and Hospitality Pipeline Lead: `Shamiur Rahman`
- Comparison and Visualization Lead (Integrator): `Nawang Lhantso`

**Submission filename:** `MP03_Notebook_team_15.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

In [1]:
!git clone https://github.com/nawanglhantso/cis3120-spring2026

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 87 (delta 29), reused 25 (delta 25), pack-reused 53 (from 1)
Receiving objects: 100% (87/87), 133.87 KiB | 5.35 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
%cd cis3120-spring2026

/content/cis3120-spring2026


---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [3]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 6.9 MB/s eta 0:00:00


In [4]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 15 - Shamiur.rahman1@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [5]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [6]:
import importlib
import mp03.seeds
importlib.reload(mp03.seeds)


<module 'mp03.seeds' from '/content/cis3120-spring2026/mp03/seeds.py'>

In [7]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

# Extended Financial Services phrases for MP03

FINANCIAL_SERVICES_PHRASES = (
    FINANCIAL_SERVICES_PHRASES
    + [
        "headquarters relocation",
        "new office",
        "office opening",
        "financial center",
        "service center",
        "corporate office",
        "branch network",
        "regional headquarters",
        "office expansion",
        "operations hub",
        "new headquarters",
        "office relocation",
    ]
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 14
Financial Services phrases: 28
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 10


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [8]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [9]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [10]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [11]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [12]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [13]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [14]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    Implementation hint: each EDGAR hit has hit["_source"]["tickers"];
    match case-insensitively and return only matching hits.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """
    # TODO: implement this function.
    ticker_set = {ticker.upper() for ticker in ticker_list}
    filtered_candidates = []

    for hit in candidates:
        source = hit.get("_source", {})

        hit_tickers = source.get("tickers") or []
        hit_tickers = {ticker.upper() for ticker in hit_tickers}

        display_names = source.get("display_names") or []
        display_text = " ".join(display_names).upper()

        ticker_in_display_name = any(
            f"({ticker})" in display_text for ticker in ticker_set
        )

        if ticker_set.intersection(hit_tickers) or ticker_in_display_name:
            filtered_candidates.append(hit)

    return filtered_candidates

In [15]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    ticker_list : list[str]
        Industry-specific ticker list.
    phrase_list : list[str]
        Industry-specific search-phrase list.
    window_days : int
        Length of the EDGAR date window (e.g., 30, 60, 90, 180, 360).

    Returns
    -------
    list[dict]
        One dict per geocoded location event, with the "industry" field set
        to industry_label. Records that fail classification or geocoding are
        excluded from the return value.

    Implementation guidance
    -----------------------
    1. Compute start_date and end_date from window_days.
    2. Call search_edgar_all_phrases(phrase_list, start_date, end_date).
    3. Filter the candidate list with filter_candidates_by_tickers.
    4. For each filtered candidate: fetch_exhibit_text, then extract_with_claude.
    5. Keep only records where is_location_event is True.
    6. For each kept record, geocode via geocode_location; drop records that
       fail geocoding.
    7. Add the "industry" field to each surviving record.
    """
    # TODO: implement this function.
    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    candidates= search_edgar_all_phrases(phrase_list, start_date, end_date)

    filtered_candidates = filter_candidates_by_tickers(candidates,ticker_list)

    geocoded_events = []

    for hit in filtered_candidates:
      text, url = fetch_exhibit_text(hit)
      src = hit.get("_source", {})

      filing = {
          "company": (src.get("display_names") or ["(unknown)"])[0],
          "ticker": (src.get("tickers") or [None])[0],
          "file_date": src.get("file_date"),
          "accession": hit["_id"].split(":")[0],
          "url": url,
          "text": text,
        }

      record = extract_with_claude(filing)
      if record.get("is_location_event"):
        coords = geocode_location(record.get("city"), record.get("state"))

        if coords:
          record["latitude"], record["longitude"] = coords
          record["industry"] = industry_label
          geocoded_events.append(record)


    return geocoded_events



In [16]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    # TODO: implement this function.
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": estimated_cost_usd,
    }

In [17]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [18]:
# TODO: run window trials for Financial Services.
# Financial Services window trials

fs_events_by_window = {}

for window_days in [30, 60, 90, 180, 360]:
    print(f"Running Financial Services trial for {window_days} days...")

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    fs_candidates = search_edgar_all_phrases(
        FINANCIAL_SERVICES_PHRASES,
        start_date,
        end_date,
    )

    fs_filtered_candidates = filter_candidates_by_tickers(
        fs_candidates,
        FINANCIAL_SERVICES_TICKERS,
    )

    fs_events = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=window_days,
    )
    fs_events_by_window[window_days] = fs_events


    fs_cost = sum(
        (event.get("input_tokens", 0) / 1_000_000) * 1
        + (event.get("output_tokens", 0) / 1_000_000) * 5
        for event in fs_events
    )

    fs_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window_days,
        candidate_count=len(fs_filtered_candidates),
        event_count=len(fs_events),
        estimated_cost_usd=fs_cost,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([fs_row])],
        ignore_index=True,
    )

    display(window_results)

    if len(fs_events) >= 8:
        print(f"Financial Services reached the target at {window_days} days.")
        break

Running Financial Services trial for 30 days...
  transient error on '"operations center"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22operations+center%22&dateRange=custom&startdt=2026-04-16&enddt=2026-05-16&forms=8-K&from=0. retrying in 5s...
  transient error on '"operations center"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22operations+center%22&dateRange=custom&startdt=2026-04-16&enddt=2026-05-16&forms=8-K&from=0. retrying in 10s...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0


Running Financial Services trial for 60 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0


Running Financial Services trial for 90 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0


Running Financial Services trial for 180 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0


Running Financial Services trial for 360 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0


### 4.2 Window trials — Travel and Hospitality

In [19]:
# TODO: run window trials for Travel and Hospitality (analogous to 4.1 above).
# Travel and Hospitality window trials

travel_events_by_window = {}

for window_days in [30, 60, 90, 180, 360]:

    print(f"Running Travel and Hospitality trial for {window_days} days...")

    travel_events = run_industry_pipeline(
        "Travel and Hospitality",
        TRAVEL_HOSPITALITY_TICKERS,
        TRAVEL_HOSPITALITY_PHRASES,
        window_days=window_days,
    )

    travel_events_by_window[window_days] = travel_events

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    travel_candidates = search_edgar_all_phrases(
        TRAVEL_HOSPITALITY_PHRASES,
        start_date,
        end_date,
    )

    travel_filtered_candidates = filter_candidates_by_tickers(
        travel_candidates,
        TRAVEL_HOSPITALITY_TICKERS,
    )

    travel_cost = sum(
        (event.get("input_tokens", 0) / 1_000_000) * 1
        + (event.get("output_tokens", 0) / 1_000_000) * 5
        for event in travel_events
    )

    travel_row = summarize_window_trial(
        industry_label="Travel and Hospitality",
        window_days=window_days,
        candidate_count=len(travel_filtered_candidates),
        event_count=len(travel_events),
        estimated_cost_usd=travel_cost,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([travel_row])],
        ignore_index=True,
    )

    display(window_results)

    if len(travel_events) >= 8:
        print(f"Travel and Hospitality reached the target at {window_days} days.")
        break


Running Travel and Hospitality trial for 30 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0
5,Travel and Hospitality,30,0,0,0


Running Travel and Hospitality trial for 60 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0
5,Travel and Hospitality,30,0,0,0
6,Travel and Hospitality,60,0,0,0


Running Travel and Hospitality trial for 90 days...
  transient error on '"new hotel"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+hotel%22&dateRange=custom&startdt=2026-02-15&enddt=2026-05-16&forms=8-K&from=0. retrying in 5s...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0
5,Travel and Hospitality,30,0,0,0
6,Travel and Hospitality,60,0,0,0
7,Travel and Hospitality,90,0,0,0


Running Travel and Hospitality trial for 180 days...
  transient error on '"new gateway"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+gateway%22&dateRange=custom&startdt=2025-11-17&enddt=2026-05-16&forms=8-K&from=0. retrying in 5s...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0
5,Travel and Hospitality,30,0,0,0
6,Travel and Hospitality,60,0,0,0
7,Travel and Hospitality,90,0,0,0
8,Travel and Hospitality,180,1,0,0


Running Travel and Hospitality trial for 360 days...


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0
1,Financial Services,60,0,0,0
2,Financial Services,90,0,0,0
3,Financial Services,180,0,0,0
4,Financial Services,360,1,0,0
5,Travel and Hospitality,30,0,0,0
6,Travel and Hospitality,60,0,0,0
7,Travel and Hospitality,90,0,0,0
8,Travel and Hospitality,180,1,0,0
9,Travel and Hospitality,360,3,0,0


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [20]:
# TODO: set the chosen window length and run the final pipelines.
#
CHOSEN_WINDOW_DAYS = 360

fs_events  = run_industry_pipeline(
     "Financial Services",
     FINANCIAL_SERVICES_TICKERS,
     FINANCIAL_SERVICES_PHRASES,
     window_days=CHOSEN_WINDOW_DAYS,
 )

th_events = run_industry_pipeline(
     "Travel and Hospitality",
     TRAVEL_HOSPITALITY_TICKERS,
     TRAVEL_HOSPITALITY_PHRASES,
     window_days=CHOSEN_WINDOW_DAYS,
 )

all_events = fs_events + th_events
print(f"Financial Services:    {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")

Financial Services:    0 events
Travel and Hospitality: 0 events
Total:                  0 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [21]:
# TODO: construct the integrated map.
#
m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

# choose icon color from the industry label
industry_colors = {
    "Financial Services": "navy",
    "Travel and Hospitality": "teal",
}

# choose icon name from event["event_type"]
event_icons = {
    "opening": "home",
    "closing": "times-circle",
    "relocation": "map-marker-alt",
    "expansion": "plus-circle",
    "other": "info-circle",
    None: "info-circle",
}

for event in all_events:
    lat = event.get("latitude")
    lon = event.get("longitude")

    if lat is None or lon is None:
        continue

#build popup HTML containing company, ticker, industry, filing date,

    popup_html = f"""
    <b>Company:</b> {event.get("company", "Unknown")}<br>
    <b>Ticker:</b> {event.get("ticker", "N/A")}<br>
    <b>Industry:</b> {event.get("industry", "Unknown")}<br>
    <b>Filing date:</b> {event.get("file_date", "N/A")}<br>
    <b>Event type:</b> {event.get("event_type", "other")}<br>
    <b>Summary:</b> {event.get("summary", "No summary available")}<br>
    <a href="{event.get("url", "#")}" target="_blank">Open SEC filing</a>
    """

    marker = folium.Marker(
        location=[event["latitude"], event["longitude"]],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=industry_colors.get(event.get("industry"), "gray"),
            icon=event_icons.get(event.get("event_type"), "info-circle"),
            prefix="fa"
        ),
    )

    marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [22]:
# TODO: export the rendered map to the required path.
import os

os.makedirs("../maps", exist_ok=True)

OUTPUT_PATH = "../maps/mp03_map_team_15.html"

m.save(OUTPUT_PATH)

print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_15.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale


### 6.2 Search-phrase rationale



### 6.3 Window-experiment results


### 6.4 Stage 3 classification quality per industry

*TODO: For each industry, document observed precision and any patterns in the Stage 3 classifications (false positives, false negatives, ambiguous cases). Use small numerical examples where possible.*


### 6.5 Limitations

*TODO: Identify limitations the team encountered and discuss how each affects the comparative reflection.*

In [23]:
import os

In [24]:
os.makedirs("../methodology", exist_ok=True)
os.makedirs("../reflections", exist_ok=True)

In [28]:
methodology_text = """
# 6.Methodology

##6.1 Ticker-list rationale
Shamiur — Travel and Hospitality Pipeline Lead:
The seeded Travel and Hospitality ticker list already contained many major hotel, resort, airline, cruise, and travel-related companies.
During the early window-tuning trials, the original ticker list successfully returned EDGAR filings related to travel operations and expansion activity.
Because the seeded list already provided broad industry coverage, only minor adjustments were necessary, and the team focused more heavily on improving the search phrases.

Mahi - Financial Services Pipeline Lead -
I expanded the Financial Services ticker list by adding companies from the capital markets and brokerage segment such as Goldman Sachs, Morgan Stanley, and Charles Schwab.
The original seeded list focused mainly on banks, insurance, and payments, so these additions helped broaden industry coverage and improve the chances of identifying location related events across multiple Financial Services sub sectors.
These additions helped broaden industry coverage and improve the chances of identifying location related events across multiple Financial Services sub sectors.

##6.2 Search-phrase rationale
Shamiur - Travel and Hospitality Pipeline Lead:
The original Travel and Hospitality phrases were not producing any results during the window trials. Therefore, I modified it by adding 6 more phrases and began achieving results from there.
To improve recall,additional phrases related to hotel openings, resort expansion, airport expansion,and route expansion were added.
After extending the phrase list, the pipeline began returning home more relevant candidate filings and location event classification.

Mahi - Financial Services Pipeline Lead:
The Financial Services search phrases were expanded by adding terms such as “headquarters relocation,” “new office,” “office opening,” “financial center,” and “corporate office.”
The seeded phrases initially focused primarily on branch related activity, so the additional phrases helped capture a wider range of location
related events involving office expansions, relocations, and operations centers.

##6.3 Window-experiment results
The project required testing multiple multiple EDGAR search windows in order to determine the smallest window
capable of generating sufficient location events counts while remaining below the assignment
cumulative cost limit of $3.00.
The team tested all given window lengths of 30,60,90,180, and 360 days.
Although increasing the search window produced more candidate filings,
the Financial Services pipeline still generated very few valid location events
after Stage 3 classification. Many filings referenced office locations or
headquarters addresses only incidentally rather than announcing genuine openings,
relocations, expansions, or closures. As a result, the final number of mapped
Financial Services events remained extremely limited, including some windows
that produced zero valid events.

##6.4 Stage 3 classification quality per industry remained below the ideal threshold of eight events, the assignment
Travel and Hospitality filings generally produced stronger classification precision because many filings explicitly
referenced hotel openings, resort expansions, airport improvements, or new travel routes.
These filings typically contained direct geographic references and clear descriptions of physical expansion activity,
which improved the accuracy of Stage 3 extraction and classification.
Financial Services filings generated more false positives because many 8-K filings mentioned office addresses,
branch locations, or headquarters information only in boilerplate language rather than announcing genuine physical
location events. Some filings referenced locations incidentally in operational updates, earnings materials, or regulatory
disclosures, causing them to be filtered out during classification.
The team also observed ambiguous cases where filings discussed operational restructuring or regional consolidation without
clearly indicating whether a facility was opening, closing, relocating, or expanding. These ambiguous cases lowered classification
confidence and reduced the number of events ultimately retained for mapping.

##6.5 Limitations
One major limitation of the project was the relatively small number of valid location event filings available within the selected
industries, particularly Financial Services. Even after expanding ticker lists and search phrases, some windows still produced sparse
event counts. This reduced the density of the final comparison map and limited the strength of geographic conclusions.
Another limitation involved EDGAR full text search behavior. Some filings matched the search phrases but did not actually describe meaningful
location events after Stage 3 classification. Broader search phrases improved recall but also introduced additional false positives,
creating a tradeoff between retrieval breadth and classification precision.
The project was also limited by automated extraction and geocoding. Some filings lacked sufficiently detailed geographic information for
reliable location extraction, while others could not be geocoded successfully. As a result, potentially relevant filings may have been excluded
from the final integrated map.At the 360 day window, one or both industries still remained below the target of eight location events.
This shortfall is acceptable under the assignment instructions because the maximum required window was tested. However, the low event count means
that the comparative reflection and geographic conclusions should be interpreted cautiously.
"""

In [29]:
with open("../methodology/mp03_methodology_team_15.md", "w") as f:
    f.write(methodology_text)

print("Methodology file created.")

Methodology file created.


---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

In [30]:
reflection_text = """
## Comparative Reflection
The geographic patterns observed in the project reveal important differences in
how the Financial Services industry and the Travel and Hospitality industry deploy
physical capacity. Travel and Hospitality companies tend to expand through highly
visible physical infrastructure such as hotels, resorts, airport routes, and
destination-based properties. As a result, their filings more frequently referenced
specific geographic expansions and operational growth tied directly to customer demand
and tourism activity.
Financial Services companies, in contrast, produced very few valid locationevent
filings after Stage 3 classification because many filings referenced office addresses,
headquarters locations, or operational centers only incidentally rather than announcing
genuine openings, relocations, expansions, or closures. The industry also increasingly
operates through digital platforms and centralized operations rather than frequent
physical expansion.
Many Financial Services firms are consolidating office space, regional hubs, and
branch networks as online banking and digital investment platforms become more
dominant. This helps explain why the Travel and Hospitality pipeline generally
produced clearer and more numerous location-event classifications compared to
Financial Services.
The differences between the industries also reflect their underlying business economics.
Travel and Hospitality firms depend heavily on physical presence and geographic
accessibility because their revenue depends on customer travel, tourism demand,
and destination capacity. Financial Services firms rely more heavily on technology
infrastructure, digital delivery systems, and centralized operational efficiency,
reducing the importance of frequent physical expansion announcements.
The project also highlighted the limitations of relying on SEC 8-K filings for
geographic analysis. Many filings referenced locations only incidentally, especially
within Financial Services, leading to lower precision during Stage 3 classification.
Additionally, sparse event counts at smaller windows limited the density of the
final comparison map. Despite these limitations, the project still demonstrated
meaningful differences between the industries and provided insight into how
sector-specific economics influence geographic expansion and operational strategy.
"""

with open("../reflections/mp03_reflection_team_15.md", "w") as f:
    f.write(reflection_text)

print("Reflection file created.")

Reflection file created.


---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.